# JSON Output Parser and Pydantic Output Parser in LangChain

## 1. Why Do We Need These Parsers?

An LLM naturally produces **text**.

For example:

```text
The product is a MacBook Pro. It costs $1999 and is currently available.
```

But an application may need structured data:

```json
{
  "product": "MacBook Pro",
  "price": 1999,
  "available": true
}
```

This is where **output parsers** become useful.

### Basic Flow

```text
User Input
    ↓
Prompt Template
    ↓
LLM
    ↓
Raw Text
    ↓
Output Parser
    ↓
Structured Data
```

Two important parsers are:

* `JsonOutputParser`
* `PydanticOutputParser`

---

# 2. JSON Output Parser

## Definition

`JsonOutputParser` is a LangChain output parser that helps parse an LLM response into **JSON-compatible structured data**.

It is useful when your application expects fields such as:

```json
{
  "name": "Arun",
  "age": 27,
  "skills": ["Python", "LangChain"]
}
```

instead of free-form text.

---

# 3. Import `JsonOutputParser`

```python
from langchain_core.output_parsers import JsonOutputParser
```

Create the parser:

```python
parser = JsonOutputParser()
```

---

# 4. Basic JSON Output Example

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_template(
    """
    Extract the following information from the text.

    Return the result as JSON.

    Text:
    {text}
    """
)

chain = prompt | model | parser

response = chain.invoke({
    "text": """
    Arun is 27 years old.
    He knows Python, LangChain and React.
    """
})

print(response)
```

Conceptually, the result is:

```python
{
    "name": "Arun",
    "age": 27,
    "skills": [
        "Python",
        "LangChain",
        "React"
    ]
}
```

---

# 5. What Does `JsonOutputParser` Do?

Think of it as:

```text
LLM Output
    ↓
JSON Parser
    ↓
Python Dictionary / JSON-compatible object
```

For example:

```text
LLM:

{
    "name": "Arun",
    "age": 27
}

        ↓

JsonOutputParser

        ↓

{
    "name": "Arun",
    "age": 27
}
```

Now your application can access:

```python
response["name"]
response["age"]
```

---

# 6. Important Limitation of JSON Output Parser

A basic `JsonOutputParser` primarily focuses on **JSON formatting/parsing**.

It doesn't provide the same level of application-level schema validation as a Pydantic model.

For example, you might expect:

```json
{
    "name": "Arun",
    "age": 27
}
```

But the model could potentially return:

```json
{
    "name": "Arun",
    "age": "twenty seven"
}
```

The JSON itself can be valid JSON, even though `age` is not the type your application expects.

That's where **PydanticOutputParser** becomes useful.

---

# 7. Pydantic Output Parser

## Definition

`PydanticOutputParser` is a LangChain parser that uses a **Pydantic model as the output schema**.

It provides:

* Structured output
* Type definitions
* Schema validation
* Pydantic model objects

---

# 8. Creating a Pydantic Schema

```python
from pydantic import BaseModel


class Student(BaseModel):
    name: str
    age: int
    skills: list[str]
```

This defines:

```text
Student
├── name → string
├── age → integer
└── skills → list of strings
```

---

# 9. Create `PydanticOutputParser`

```python
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(
    pydantic_object=Student
)
```

Now the parser knows the expected structure.

---

# 10. Complete Pydantic Output Parser Example

```python
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


class Student(BaseModel):
    name: str
    age: int
    skills: list[str]


model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


parser = PydanticOutputParser(
    pydantic_object=Student
)


prompt = ChatPromptTemplate.from_template(
    """
    Extract student information from the following text.

    {format_instructions}

    Text:
    {text}
    """
)


chain = prompt | model | parser


response = chain.invoke({
    "text": """
    Arun is 27 years old.
    He knows Python, LangChain and React.
    """,
    "format_instructions": parser.get_format_instructions()
})


print(response)
```

The result is conceptually:

```python
Student(
    name="Arun",
    age=27,
    skills=[
        "Python",
        "LangChain",
        "React"
    ]
)
```

---

# 11. What is `get_format_instructions()`?

This is an important concept.

When using `PydanticOutputParser`, you can get instructions describing the expected output format:

```python
parser.get_format_instructions()
```

These instructions can be inserted into the prompt:

```python
prompt = ChatPromptTemplate.from_template(
    """
    Extract information.

    {format_instructions}

    Text:
    {text}
    """
)
```

The model receives instructions describing the required structure.

### Flow

```text
Pydantic Model
      ↓
PydanticOutputParser
      ↓
get_format_instructions()
      ↓
Prompt
      ↓
LLM
      ↓
JSON-like Response
      ↓
PydanticOutputParser
      ↓
Validated Pydantic Object
```

---

# 12. Why Do We Need `get_format_instructions()`?

Suppose your Pydantic schema is:

```python
class Student(BaseModel):
    name: str
    age: int
    skills: list[str]
```

The LLM needs to know what format you expect.

`get_format_instructions()` generates instructions based on the schema.

Conceptually, the prompt tells the model:

```text
Return the result in the required JSON structure.

Fields:
- name: string
- age: integer
- skills: array of strings
```

So the model has a better understanding of the expected output.

---

# 13. JSON Parser vs Pydantic Parser

This is one of the most important distinctions.

| Feature           | `JsonOutputParser`     | `PydanticOutputParser`            |
| ----------------- | ---------------------- | --------------------------------- |
| Output            | JSON-compatible data   | Pydantic object                   |
| Schema            | Basic/optional         | Pydantic schema                   |
| Type validation   | Limited                | Yes                               |
| Field definitions | Less strict            | Explicit                          |
| Nested models     | Possible               | Very convenient                   |
| Python type hints | No                     | Yes                               |
| Validation        | JSON parsing           | Pydantic validation               |
| Best for          | Simple structured JSON | Strongly defined application data |

---

# 14. Example Comparison

## JSON Parser

```python
parser = JsonOutputParser()
```

Result:

```python
{
    "name": "Arun",
    "age": 27
}
```

Access:

```python
response["name"]
```

---

## Pydantic Parser

```python
class Student(BaseModel):
    name: str
    age: int


parser = PydanticOutputParser(
    pydantic_object=Student
)
```

Result:

```python
Student(
    name="Arun",
    age=27
)
```

Access:

```python
response.name
response.age
```

---

# 15. JSON Parser — Practical Application

Imagine you're building a **sentiment analysis application**.

You want:

```json
{
    "sentiment": "positive",
    "confidence": 0.92,
    "reason": "The customer is satisfied."
}
```

You could use:

```python
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
```

Then:

```python
chain = prompt | model | parser
```

Result:

```python
{
    "sentiment": "positive",
    "confidence": 0.92,
    "reason": "The customer is satisfied."
}
```

Your application can then:

```python
if response["sentiment"] == "positive":
    print("Happy customer")
```

---

# 16. Pydantic Parser — Practical Application

Suppose you're building a **resume parser**.

Define:

```python
class Candidate(BaseModel):
    name: str
    experience: float
    skills: list[str]
    email: str
```

Then:

```python
parser = PydanticOutputParser(
    pydantic_object=Candidate
)
```

The model response becomes a validated object:

```python
Candidate(
    name="Arun",
    experience=2.0,
    skills=["Python", "LangChain", "React"],
    email="arun@example.com"
)
```

This is much easier to work with in Python application code.

---

# 17. Nested Pydantic Models

Pydantic becomes particularly useful when the output is complex.

```python
from pydantic import BaseModel


class Address(BaseModel):
    city: str
    country: str


class Candidate(BaseModel):
    name: str
    age: int
    skills: list[str]
    address: Address
```

Expected structure:

```text
Candidate
│
├── name
├── age
├── skills
│
└── address
    ├── city
    └── country
```

The model can produce:

```python
Candidate(
    name="Arun",
    age=27,
    skills=["Python", "LangChain"],
    address=Address(
        city="Delhi",
        country="India"
    )
)
```

---

# 18. Output Parser vs `with_structured_output()`

This distinction is **very important in modern LangChain**.

You may see older/tutorial-style code like:

```python
parser = PydanticOutputParser(
    pydantic_object=Student
)

chain = prompt | model | parser
```

But LangChain also provides:

```python
structured_model = model.with_structured_output(Student)
```

Then:

```python
response = structured_model.invoke(
    "Arun is 27 and studying MCA."
)
```

### Conceptual Difference

### `PydanticOutputParser`

```text
Prompt
  ↓
LLM
  ↓
Model-generated output
  ↓
PydanticOutputParser
  ↓
Pydantic object
```

### `with_structured_output()`

```text
Prompt
  ↓
LLM + Structured Output Mechanism
  ↓
Structured result
  ↓
Pydantic object
```

`with_structured_output()` can use the model/provider's supported structured-output or tool-calling capabilities, rather than relying solely on instructions in the prompt followed by local parsing.

### Interview Point ⭐

> `PydanticOutputParser` is a parser-based approach, while `with_structured_output()` is a model-level structured-output interface and is often preferable when the selected model/provider supports it.

---

# 19. When Should You Use Which?

## Use `JsonOutputParser` when:

* You need simple JSON output.
* Your downstream code works with dictionaries.
* You don't need strict Python schema validation.
* The output structure is relatively simple.

Example:

```text
LLM
 ↓
JSON
 ↓
Dictionary
```

---

## Use `PydanticOutputParser` when:

* You want a strongly defined schema.
* You want Python type hints.
* You need validation.
* You have nested/complex data.
* Your application works naturally with Pydantic models.

Example:

```text
LLM
 ↓
Pydantic Parser
 ↓
Validated Model
```

---

## Use `with_structured_output()` when:

* Your model/provider supports structured output.
* You want schema-based model responses.
* You want to avoid manually constructing format instructions where possible.
* You're building a modern production LangChain application.

---

# 20. Complete Comparison

```text
                 LLM
                  │
        ┌─────────┴─────────┐
        │                   │
        ↓                   ↓
 JsonOutputParser     PydanticOutputParser
        │                   │
        ↓                   ↓
    Dictionary        Pydantic Object
        │                   │
        ↓                   ↓
Simple Processing     Validation + Types
```

And with modern structured output:

```text
                 Chat Model
                     │
                     ↓
       with_structured_output()
                     │
                     ↓
              Schema / Model
                     │
                     ↓
             Structured Result
```

---

# 21. Interview Questions & Answers

## Beginner

### 1. What is `JsonOutputParser`?

`JsonOutputParser` is a LangChain output parser used to parse model responses into JSON-compatible structured data.

---

### 2. What is `PydanticOutputParser`?

`PydanticOutputParser` parses an LLM response according to a Pydantic schema and returns a validated Pydantic object.

---

### 3. What is Pydantic?

Pydantic is a Python library that uses type hints and models to define and validate structured data.

---

### 4. What does `get_format_instructions()` do?

It generates formatting instructions based on the Pydantic schema so they can be included in the prompt sent to the model.

---

## Intermediate

### 5. What is the main difference between JSON and Pydantic output parsing?

`JsonOutputParser` focuses primarily on parsing JSON-compatible output, while `PydanticOutputParser` adds a predefined Python schema and validation through Pydantic.

---

### 6. Why is Pydantic better for complex structures?

Pydantic supports:

* Type hints
* Nested models
* Required fields
* Optional fields
* Validation
* Clear schemas

This makes it well suited for complex application data.

---

### 7. What does this chain do?

```python
chain = prompt | model | parser
```

It passes:

```text
Prompt Output
     ↓
LLM
     ↓
Parser
```

Each component's output becomes the next component's input.

---

### 8. What is the difference between `PydanticOutputParser` and `with_structured_output()`?

`PydanticOutputParser` is an explicit output-parsing approach that parses model-generated output according to a Pydantic schema.

`with_structured_output()` is a model interface that uses the model/provider's supported structured-output capabilities and returns data according to the specified schema.

---

# 22. Scenario-Based Questions

### 9. You are extracting product information from 10,000 documents. What would you use?

If the output is simple and only needs JSON:

```text
JsonOutputParser
```

If the application requires strong schema validation:

```text
PydanticOutputParser
```

For a modern supported model, I'd also consider:

```text
with_structured_output()
```

---

### 10. Your output needs nested customer and address objects. What would you choose?

A Pydantic schema is a strong choice because nested Pydantic models make the structure explicit and validated.

---

### 11. Your application only needs a simple dictionary. Do you need Pydantic?

Not necessarily. `JsonOutputParser` may be sufficient if strict schema validation isn't required.

---

### 12. The LLM keeps returning invalid JSON. What can you do?

Possible approaches include:

* Improve the prompt.
* Use format instructions.
* Use a JSON/structured-output mechanism.
* Use schema validation.
* Add retry/error-handling logic.
* Prefer model-native structured output when supported.

---

# Key Takeaways

* **Output Parser** converts LLM output into a format your application can use.
* `JsonOutputParser` is useful for **JSON-compatible structured data**.
* `PydanticOutputParser` uses a **Pydantic schema** to parse and validate output.
* `BaseModel` defines the Pydantic schema.
* `get_format_instructions()` provides formatting instructions for parser-based approaches.
* JSON output is generally accessed like:

```python
response["name"]
```

* Pydantic output is generally accessed like:

```python
response.name
```

* For complex application data, Pydantic provides stronger structure and validation.
* Modern LangChain also provides:

```python
model.with_structured_output(MySchema)
```

which can be preferable when the model/provider supports structured output.

## ⭐ Interview One-Liner

> **`JsonOutputParser` converts an LLM response into JSON-compatible data, while `PydanticOutputParser` parses the response according to a Pydantic schema and provides typed, validated Python objects.**

### Mental Model

```text
                 PROMPT
                    ↓
             ChatPromptTemplate
                    ↓
                 LLM
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
 JsonOutputParser     PydanticOutputParser
          ↓                   ↓
    Dictionary          Pydantic Object
          ↓                   ↓
   Simple Processing   Type + Validation
```

**Most important progression to remember:**

```text
StrOutputParser
      ↓
Plain String

JsonOutputParser
      ↓
JSON / Dictionary

PydanticOutputParser
      ↓
Validated Pydantic Object

with_structured_output()
      ↓
Model-level Structured Output
```
